In [1]:
!pip install -q google-genai google-adk pydantic scikit-learn pandas
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [25]:
import time
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from google import genai
from google.genai import types
from sklearn.metrics import accuracy_score, cohen_kappa_score

MODEL = "gemini-3.5-flash-lite"
client = genai.Client()

class ClaimCheck(BaseModel):
    claim: str
    evidence_quote: str
    verdict: Literal["supported", "unsupported", "contradicted"]

class VerifierReport(BaseModel):
    claim_checks: list[ClaimCheck]
    passed: bool
    reading_level_match: bool
    flagged_passages: list[str]

PROMPT = """You are a clinical faithfulness verifier. Judge the TEXT against the SOURCE only.

The SOURCE is a set of short excerpts from the papers the TEXT cites. The papers may
contain more than the excerpts show, so extra background detail is not an error.

SOURCE:
{source}

TEXT TO AUDIT:
{text}

1. Identify the main claim of the TEXT: the central point it makes. Ignore framing words
   like "Additionally" or "It is important to note".
2. Label each claim:
   supported = the source states it, entails it, or says the same thing in different words.
   contradicted = the source says something different or opposite. Check direction
                  (increase vs decrease), numbers, and whether an effect persists or stops.
                  Also contradicted: the TEXT states as definite what the source says is
                  uncertain, inconsistent or "may" happen; claims an effect that the source
                  says was not significant or happened "only" for something else; or says
                  the opposite of a negative finding ("does not", "no effect").
   unsupported = no quote found, including general background or definitions the source
                 never states.
3. passed = true only if BOTH hold:
   a) the main claim is supported by the source (the source must actually address the
      same topic and point, not merely mention related words), and
   b) no claim is contradicted.
   Peripheral detail that is merely unsupported does not make passed false.
4. reading_level_match = true if the wording fits roughly an 8th-grade reading level.
5. flagged_passages = exact text of any contradicted claim, plus the main claim if unsupported."""

def judge(source, text):
    for attempt in range(5):
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=PROMPT.format(source=source, text=text),
                config=types.GenerateContentConfig(
                    temperature=0, response_mime_type="application/json",
                    response_schema=VerifierReport),
            )
            return VerifierReport.model_validate_json(r.text)
        except Exception as e:
            if attempt == 4: raise
            time.sleep(2 ** attempt * 5)

def calibrate(df):
    """df needs: source_text, answer_text, human_label (0/1)"""
    preds = [int(judge(r.source_text, r.answer_text).passed) for r in df.itertuples()]
    y = df["human_label"].astype(int)
    print("accuracy:", accuracy_score(y, preds))
    print("kappa:", cohen_kappa_score(y, preds))
    print("false positives:", sum(p == 1 and t == 0 for p, t in zip(preds, y)))
    return df.assign(verifier_label=preds)

In [3]:
## synthetic test case
r = judge("Metformin 500 mg twice daily for type 2 diabetes.",
          "You take a diabetes pill called metformin once a day.")
print(r.passed, r.flagged_passages)

False ['once a day']


**Testing on MEDAesqa**

In [4]:
import json, re
import pandas as pd
from google.colab import files

uploaded = files.upload()
fname = list(uploaded.keys())[0]

with open(fname) as f:
    data = json.load(f)

rows = []
for q in data:
    for mid, ans in q["machine_generated_answers"].items():
        for s in ans["answer_sentences"]:
            ca = s["citation_assessment"]
            if not ca:
                continue
            ev = [c["evidence_support"] for c in ca if c["evidence_support"]]
            if not ev:
                continue
            rels = {c["evidence_relation"] for c in ca}
            rows.append({
                "question_id": q["question_id"],
                "method": mid,
                "source_text": "\n".join(ev),
                "answer_text": re.sub(r"\s*\[[\d,\s]+\]", "", s["answer_sentence"]).strip(),
                "human_label": int("supporting" in rels and "contradicting" not in rels),
            })

df = pd.DataFrame(rows)
print(len(df), "rows")
print(df.human_label.value_counts())

Saving medaesqa_v1.json to medaesqa_v1 (3).json
3610 rows
human_label
1    3232
0     378
Name: count, dtype: int64


In [5]:
df.head()

,question_id,method,source_text,answer_text,human_label
0,116,M1,Dietary modifications may be an effective tool...,There are several natural ways to prevent and ...,1
1,116,M1,Avoiding the supine position during sleep and ...,"Lifestyle modifications such as weight loss, a...",1
2,116,M1,This report describes the chiropractic managem...,"Dietary modifications, such as those described...",1
3,116,M1,OAT has been recommended as an appropriate fir...,"Additionally, oral appliances, conditioning to...",1
4,116,M1,Electrical stimulation of the hypoglossal nerv...,"Furthermore, alternative treatments such as hy...",1


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score

df = df.drop(columns="method")



In [9]:
before = len(df)
df = df.drop_duplicates(subset=["question_id", "source_text", "answer_text"]).reset_index(drop=True)
print(before, "->", len(df), "rows")
print(df.human_label.value_counts())

3610 -> 3526 rows
human_label
1    3161
0     365
Name: count, dtype: int64


In [10]:
train_q, test_q = train_test_split(df["question_id"].unique(), test_size=0.5, random_state=42)
train_df = df[df["question_id"].isin(train_q)].reset_index(drop=True)
test_df = df[df["question_id"].isin(test_q)].reset_index(drop=True)

In [26]:
pos = train_df[train_df.human_label == 1].sample(10, random_state=1)
neg = train_df[train_df.human_label == 0].sample(10, random_state=1)
train_sample = pd.concat([pos, neg])

results = calibrate(train_sample)
results.to_csv("train_results.csv", index=False)

accuracy: 0.85
kappa: 0.7
false positives: 2


In [27]:
pd.set_option("display.max_colwidth", 250)
wrong = results[results.human_label != results.verifier_label]
wrong[["source_text", "answer_text", "human_label", "verifier_label"]]

,source_text,answer_text,human_label,verifier_label
979,"Total elbow arthroplasty is currently an established surgical treatment for several pathologies of the elbow.\nEach year, about 80 elbow arthroplasties are performed in Denmark. Approximately two thirds are done due to rheumatoid arthritis, the o...","Surgical treatment may be necessary in some cases, and options include arthroscopic debridement and synovectomy, capsular release, ulnohumeral arthroplasty, interpositional arthroplasty, total elbow replacement, total elbow allografts, and elbow ...",1,0
676,"Plasma nitrotyrosine and malondialdehyde (MDA), representative byproducts of protein and lipid oxidative damage, were determined by enzyme immunoassa\nPlasma nitrotyrosine and MDA were increased, but only nitrotyrosine was significantly higher (p...","PCOS patients have higher levels of oxidative stress markers, such as malondialdehyde, and lower levels of antioxidants, such as glutathione, compared to healthy women.",0,1
248,"Cortisol is increasingly included in examinations of mindfulness intervention effects as an indicator of efficacy; however, the association of cortisol and mindfulness has yet to be rigorously evaluated.\nMindfulness may influence cortisol, but f...","Mindfulness-based stress reduction, meditation, and possibly mindfulness, can help lower cortisol levels.",0,1
